In [81]:
import pandas as pd
import numpy as np

# Methodology

Sentiment labels are assigned to each news article based on the sign of this aggregated `three-day excess return`. This excess return is calculated from the day a news article is first published and extends over the two subsequent days. To elaborate, excess return is defined as the difference between the return of a particular stock and the overall market return on the same day. This calculation is not limited to the day the news is published; instead, it aggregates the returns for the following two days as well, providing a
comprehensive three-day outlook.

A positive aggregated excess return leads to a sentiment label of `1`, indicating a positive sentiment. Conversely, a non-positive aggregated excess return results in a sentiment label of `0`, suggesting a negative sentiment.

In [83]:
stocks = pd.read_csv("./Data Updated/stocks_data.csv", header=[0, 1], index_col=0)
stocks_info = pd.read_csv("./Data Updated/SPY_companies_info.csv")
# Separate DataFrames by the top-level column (first row of headers)
stocks_dict = {key: stocks[key] for key in stocks.columns.levels[0]}
market = pd.read_csv("./Data Updated/market_data.csv")
market = market.set_index('Date')

In [84]:
# Calculate excess return sentiment label
def excess_return_sentiment_label(stock_data, market_data, num_days=3, price_used="Adj Close"):
    if price_used not in stock_data.columns or price_used not in market_data.columns:
        raise ValueError(f"Column '{price_used}' is missing in stock or market data.")

    stock_df = stock_data[[price_used]].copy()
    market_df = market_data[[price_used]].copy()
    market_df = market_df.add_prefix("SPY_")
    merged_df = stock_df.join(market_df)
    merged_df = merged_df.pct_change()


    merged_df["excess_returns"] = merged_df["Adj Close"] - merged_df["SPY_Adj Close"]
    merged_df["X_days_excess_returns"] =\
        (
            (1 + merged_df["excess_returns"])
            .rolling(num_days)
            .apply(lambda x: x.cumprod()[-1], raw=True)
            .shift(-(num_days - 1))
            - 1
        )

    # Label for 1 excess returns > 0, 0 otherwise
    merged_df["sentiment_label"] = (merged_df["X_days_excess_returns"] > 0).astype(int)
    merged_df = merged_df.reset_index()
    merged_df = merged_df[["Date", "sentiment_label"]]
    return merged_df


ticker_senti_date = pd.DataFrame()

for ticker in stocks_dict:
    result = excess_return_sentiment_label(stocks_dict[ticker], market)
    result["Ticker"] = ticker
    ticker_senti_date = pd.concat([ticker_senti_date, result], ignore_index=True)

ticker_senti_date.dropna(subset=["sentiment_label"])
ticker_senti_date

,Date,sentiment_label,Ticker
0,2014-01-02,0,A
1,2014-01-03,1,A
2,2014-01-06,1,A
3,2014-01-07,1,A
4,2014-01-08,1,A
...,...,...,...
1381736,2024-11-22,0,ZTS
1381737,2024-11-25,0,ZTS
1381738,2024-11-26,0,ZTS
1381739,2024-11-27,0,ZTS


In [91]:
# Process train and test set
filtered_news_train_df = pd.read_csv("./Data Updated/filtered_news_train_df.csv")
filtered_news_test_df = pd.read_csv("./Data Updated/filtered_news_test_df.csv")

news_train_combined_df = pd.merge(
    filtered_news_train_df, ticker_senti_date,
    left_on=["date", "ticker"],
    right_on=["Date", "Ticker"],
    how="left"
)
news_train_combined_df = news_train_combined_df.drop(["date", "ticker"], axis=1)
news_train_combined_df.dropna(subset=["Date"], inplace=True)

news_test_combined_df = pd.merge(
    filtered_news_test_df, ticker_senti_date,
    left_on=["date", "ticker"],
    right_on=["Date", "Ticker"],
    how="left"  # Use 'left' to keep all rows from df1
)
news_test_combined_df = news_test_combined_df.drop(["date", "ticker"], axis=1)
news_test_combined_df.dropna(subset=["Date"], inplace=True)

In [92]:
# Format
train_df = news_train_combined_df
train_df["date"] = pd.to_datetime(train_df["Date"])
train_df['sentiment_label'] = train_df['sentiment_label'].astype(int)

test_df = news_test_combined_df
test_df["date"] = pd.to_datetime(test_df["Date"])
test_df['sentiment_label'] = test_df['sentiment_label'].astype(int)

In [93]:
# Further split for train and val set
val_df = train_df[train_df['date'].dt.year > 2021].reset_index(drop=True)
train_df = train_df[train_df['date'].dt.year <= 2021].reset_index(drop=True)

In [94]:
train_df

,Unnamed: 0,title,source,Date,sentiment_label,Ticker,date
0,0,Agilent Technologies Introduces New Version of...,Business Wire,2014-01-06,1,A,2014-01-06
1,1,Agilent Technologies Introduces ICP-MS and MP-...,Business Wire,2014-01-06,1,A,2014-01-06
2,2,Agilent Technologies Inc. Introduces New Exter...,Business Wire,2014-01-08,1,A,2014-01-08
3,3,AT4 Wireless Selects Agilent Technologies Test...,Business Wire,2014-01-09,1,A,2014-01-09
4,4,Agilent Technologies Introduces First USB 3.0 ...,Other,2014-01-09,1,A,2014-01-09
...,...,...,...,...,...,...,...
261602,332857,Zoetis Inc. Presents at 30th Annual Credit Sui...,PR Newswire; Business Wire; GlobeNewswire; Com...,2021-11-09,1,ZTS,2021-11-09
261603,332858,Zoetis Inc. Presents at 2021 HMG Live! Pacific...,GlobeNewswire; Company Website,2021-11-18,1,ZTS,2021-11-18
261604,332859,Zoetis Inc. Declares Dividend for the First Qu...,Business Wire,2021-12-07,1,ZTS,2021-12-07
261605,332860,Zoetis Inc. announces an Equity Buyback for $...,Capital IQ Buybacks Database,2021-12-07,1,ZTS,2021-12-07


In [96]:
train_df.to_csv("./Data Updated/train_df.csv", index=False)

In [95]:
val_df

,Unnamed: 0,title,source,Date,sentiment_label,Ticker,date
0,764,"Agilent Technologies, Inc., $ 0.21, Cash Divid...",Financial Times,2022-01-03,0,A,2022-01-03
1,765,"Agilent Technologies, Inc. Presents at Goldman...",PR Newswire; Business Wire; Company Website,2022-01-06,0,A,2022-01-06
2,766,"Agilent Technologies, Inc. Presents at JPMorga...",PR Newswire; Business Wire; Other; GlobeNewswi...,2022-01-11,1,A,2022-01-11
3,767,Agilent Announces the Innovative Seahorse XF P...,Business Wire,2022-01-24,0,A,2022-01-24
4,768,Biofidelity Ltd. announced that it has receive...,Capital IQ Transaction Database,2022-02-01,1,A,2022-02-01
...,...,...,...,...,...,...,...
64080,332971,"Zoetis Inc., Q3 2023 Earnings Call, Nov 02, 2023",Business Wire,2023-11-02,1,ZTS,2023-11-02
64081,332972,Zoetis Inc. Provides Earnings Guidance for the...,Business Wire,2023-11-02,1,ZTS,2023-11-02
64082,332973,Zoetis Inc. Reports Earnings Results for the T...,S&P Capital IQ Financials Database,2023-11-02,1,ZTS,2023-11-02
64083,332974,Zoetis Inc. Presents at Piper Sandler 35th Ann...,PR Newswire; Business Wire; Other; GlobeNewswi...,2023-11-28,0,ZTS,2023-11-28


In [97]:
val_df.to_csv("./Data Updated/val_df.csv", index=False)

In [70]:
test_df

,Unnamed: 0,title,source,Date,sentiment_label,Ticker,date
0,101,Agilent Technologies Introduces New Version of...,Business Wire,2014-01-06,1.0,A,2014-01-06
1,129,Agilent Technologies Introduces ICP-MS and MP-...,Business Wire,2014-01-06,1.0,A,2014-01-06
2,321,Agilent Technologies Inc. Introduces New Exter...,Business Wire,2014-01-08,1.0,A,2014-01-08
3,441,AT4 Wireless Selects Agilent Technologies Test...,Business Wire,2014-01-09,1.0,A,2014-01-09
4,463,Agilent Technologies Introduces First USB 3.0 ...,Other,2014-01-09,1.0,A,2014-01-09
...,...,...,...,...,...,...,...
339010,369203,"Zoetis Inc., Q3 2023 Earnings Call, Nov 02, 2023",Business Wire,2023-11-02,1.0,ZTS,2023-11-02
339011,369261,Zoetis Inc. Provides Earnings Guidance for the...,Business Wire,2023-11-02,1.0,ZTS,2023-11-02
339012,369264,Zoetis Inc. Reports Earnings Results for the T...,S&P Capital IQ Financials Database,2023-11-02,1.0,ZTS,2023-11-02
339013,371596,Zoetis Inc. Presents at Piper Sandler 35th Ann...,PR Newswire; Business Wire; Other; GlobeNewswi...,2023-11-28,0.0,ZTS,2023-11-28


In [98]:
test_df.to_csv("./Data Updated/test_df.csv", index=False)